# Generation of Random Variate

In [ ]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')


In [ ]:
# Amitava Chakraborty 22 Dec 2019

#Classical computers use pseudo-random routines
#Quantum computers can generate true random variates. 

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import sys, math, time
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from qiskit_backend_config import BasicAer
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_backend_config import execute
from qiskit.visualization import plot_histogram
q_backend = BasicAer.get_backend("qasm_simulator")

# 1. Measurement of a quantum superposition is intrinsically random

We create a quantum circuit of $n$ qubits and apply a $H-gate$ in each qubit. We then measure the circuit $m$ times.
Each measurement with produce either a $0$ or an $1$ for each qubit. Hence, in each measurement, we will get a bit-pattern of $n \mbox{bit}$. This way, after $m$ measurements, we can create a bit pattern of $m\cdot n$ bits - and that bit pattern is intrinsically random.

Suppose we need a random bit pattern of $100$ bits. We can achieve this with a quantum circuit involving $4$ qubits, measuring the circuit $25$ times.

In [ ]:
#We create a n-qubit system, here a 2-qubit one
qc = QuantumCircuit(4)
#We apply H gates to both the qubits
qc.h([0,1,2,3])
#qc.h(1)
qc.measure_all()#([0,1],[0,1])
qc.draw('mpl')

In [ ]:
# We measure the circuit m times (shots=m)
results = execute(qc, q_backend, shots=25, memory=True).result()
#We get a pure random number of n x m bits
print(results.get_memory())
bit_str = ''.join(results.get_memory())
print('Random Bit String : ',bit_str)
print('Random Bit String Length : ',len(bit_str))
plot_histogram(results.get_counts())

#Random Bits and the Bernoulli distribution
#(i.e., Bernoulli-distributed scalars, taking values either 0 or 1). 
#Starting from a simple circuit such as a Hadamard gate followed by measurement, 
#one can progress to vectors of Bernoulli-distributed elements. 
#By addition of such random variates, we could get binomial distributions. 
#By multiplication we could get geometric distributions, 
#although perhaps leading to a circuit depth that may be impractical at the moment, though.

#Let us start by importing the basic modules and creating a quantum circuit for generating random bits:

In [ ]:
#We create a quantum circuit, which is just an extension of the above circuit
#To generate a random number
def get_random_variate_circuit(num_target_qubits: int) -> QuantumCircuit:
    """
    Creates and returns a quantum circuit for random variate generation.
    :param num_target_qubits: number of qubits to be used.
    :return: quantum curcuit.
    """
    qr = QuantumRegister(num_target_qubits)
    cr = ClassicalRegister(num_target_qubits)
    circuit = QuantumCircuit(qr, cr)
    circuit.h(qr)
    circuit.barrier()
    circuit.measure(qr, cr)
    return circuit

In [ ]:
# Create and plot generating quantum circuit.
# Number of qubits utilised simultaneously.
num_qubits = 5
circuit = get_random_variate_circuit(num_qubits)
circuit.draw(output='mpl')

In [ ]:
#Uniformly-distributed scalars and vectors
#It is clear that there are many options for approximating uniformly-distributed scalars 
#by the choice of an integer from a finite range uniformly at random, 
#e.g., by a binary-code construction from the Bernoulli-distributed vectors. 

#In the following snippet, we generate random bits, 
#which we then convert using the binary-code construction, 
#up to the machine precision of a classical computer.
def get_uniform_random(circuit: QuantumCircuit, num_target_qubits: int,
                         size: int, bounds: [float,float], iInt: bool) -> np.ndarray:
    """
    Generates a vector of random float64(default)/or int values in the range bounds[min_value, max_value].
    :param circuit: quantum circuit for random variate generation.
    :param num_target_qubits: number of qubits to be used.
    :param size: length of the vector.
    :param bounds: [lower bound, upper bound]
    :param iInt: whether int values are needed
    :return: vector of random values.
    """
    vmin=float(bounds[0])
    vmax=float(bounds[1])

    # Mantissa of float64 uses 7 bytes
    float64_bits = 7 * 8                                   
    bit_str_len = (float64_bits * size + num_target_qubits - 1) // num_target_qubits
    print(bit_str_len)
    job = execute(circuit, q_backend, shots=bit_str_len, memory=True)
    bit_str = ''.join(job.result().get_memory())
    scale = float(vmax - vmin) / float(2**float64_bits - 1)
    array = np.array([vmin + scale * float(int(bit_str[i:i+float64_bits], 2))
                     for i in range(0, float64_bits * size, float64_bits)], dtype=np.float64)
    print(array)
    if iInt == True:
        array = np.rint(array).astype(np.int64)
    return array

In [ ]:
#Uniform distribution over floating point numbers - a random vector of floating-point values uniformly distributed 
#within some arbitrary selected interval:

# Draw a sample from uniform distribution.
sample = get_uniform_random(circuit, 4, size=50, bounds=[-8.2,23.3], iInt=False)

# Print out some details.
print("Uniform distribution over floating point numbers:")
print("sample type:", type(sample), ", element type:", sample.dtype, ", shape:", sample.shape)
print("sample min: {:.4f}, max: {:.4f}".format(np.amin(sample), np.amax(sample)))

# Plotting the distribution.
plt.hist(sample.ravel(),
         bins=min(int(np.ceil(np.sqrt(sample.size))), 100),
         density=True, facecolor='#0077ff', alpha=0.75)
plt.xlabel("value", size=12)
plt.ylabel("probability", size=12)
plt.title("Uniform distribution over float64 numbers in [{:.2f} ... {:.2f}]".format(
            np.amin(sample), np.amax(sample)), size=12)
plt.grid(True)
# plt.savefig("uniform_distrib_float.png", bbox_inches="tight")
plt.show()

In [ ]:
#Uniform distribution over integers - a random vector of integers:

# Draw a sample from uniform distribution.
sample = get_uniform_random(circuit, num_qubits, size=50, bounds=[11,112], iInt=True)

# Print out some details.
print("Uniform distribution over bounded integer numbers:")
print("sample type:", type(sample), ", element type:", sample.dtype, ", shape:", sample.shape)
print("sample min: {:d}, max: {:d}".format(np.amin(sample), np.amax(sample)))

# Plotting the distribution.
plt.hist(sample.ravel(),
         bins=min(int(np.ceil(np.sqrt(sample.size))), 100),
         density=True, facecolor='#00ff77', alpha=0.75)
plt.xlabel("value", size=12)
plt.ylabel("probability", size=12)
plt.title("Uniform distribution over int64 numbers in [{:d} ... {:d}]".format(
            np.amin(sample), np.amax(sample)), size=12)
plt.grid(True)
# plt.savefig("uniform_distrib_int.png", bbox_inches="tight")
plt.show()

In [ ]:
#Normal distribution
#To generate random variates with a standard normal distribution using two independent samples 
#$u_1, u_2$ of the uniform distribution on the unit interval [0, 1], 
#one can consider the Box-Muller transform to obtain a 2-vector:

#$$\begin{align}
#\begin{bmatrix}
#%R\cos(\Theta )=
#{\sqrt {-2\ln u_{1}}}\cos(2\pi u_{2}) \\
#% R\sin(\Theta )=
#{\sqrt {-2\ln u_{1}}}\sin(2\pi u_{2})
#\end{bmatrix},
#\end{align}$$
#wherein we have two independent samples of the standard normal distribution. 
#In IBM Q, this is implemented as follows:
def normal_rand_float64(circuit: QuantumCircuit, num_target_qubits: int,
                        size: int, mu: float, sigma: float) -> np.ndarray:
    """
    Draws a sample vector from the normal distribution given the mean and standard
    deviation, using the Box-Muller method. 
    """
    TINY = np.sqrt(np.finfo(np.float64).tiny)
    assert isinstance(size, int) and size > 0
    rand_vec = np.zeros((size,), dtype=np.float64)

    # Generate array of uniformly distributed samples, factor 1.5 longer that
    # actually needed.
    n = (3 * size) // 2
    x = np.reshape(get_uniform_random(circuit, num_target_qubits,
                                        2*n, [0.0, 1.0],iInt=False), (-1, 2))

    x1 = 0.0                # first sample in a pair
    c = 0                   # counter
    for d in range(size):
        r2 = 2.0
        while r2 >= 1.0 or r2 < TINY:
            # Regenerate array of uniformly distributed samples upon shortage.
            if c >= n:
                c = 0
                n = max(size // 10, 1)
                x = np.reshape(get_uniform_random(circuit, num_target_qubits,
                                                    2*n, [0.0, 1.0],iInt=False), (-1, 2))

            x1 = 2.0 * x[c, 0] - 1.0        # first sample in a pair
            x2 = 2.0 * x[c, 1] - 1.0        # second sample in a pair
            r2 = x1 * x1 + x2 * x2
            c += 1

        f = np.sqrt(np.abs(-2.0 * np.log(r2) / r2))
        rand_vec[d] = f * x1
        
    return (rand_vec * sigma + mu)

In [ ]:
#The following example demonstrates how to draw a random vector of normally distributed variates:

# Mean and standard deviation.
mu = 2.4
sigma = 5.1

# Draw a sample from the normal distribution.
start_time = time.time()
sample = normal_rand_float64(circuit, num_qubits, size=50, mu=mu, sigma=sigma)
sampling_time = time.time() - start_time

# Print out some details.
print("Normal distribution (mu={:.3f}, sigma={:.3f}):".format(mu, sigma))
print("  sample type:", type(sample), ", element type:", sample.dtype, ", shape:", sample.shape)
print("  sample min: {:.4f}, max: {:.4f}".format(np.amin(sample), np.amax(sample)))
print("  sampling time: {:.2f} secs".format(sampling_time))

In [ ]:
# Plotting the distribution.
x = np.linspace(mu - 4.0 * sigma, mu + 4.0 * sigma, 1000)
analyt = np.exp(-0.5 * ((x - mu) / sigma)**2) / (sigma * math.sqrt(2.0 * math.pi))
plt.hist(sample.ravel(),
         bins=min(int(np.ceil(np.sqrt(sample.size))), 100),
         density=True, facecolor='r', alpha=0.75)
plt.plot(x, analyt, '-b', lw=1)
plt.xlabel("value", size=12)
plt.ylabel("probability", size=12)
plt.title("Normal distribution: empirical vs analytic", size=12)
plt.grid(True)
# plt.savefig("normal_distrib.png", bbox_inches="tight")
plt.show()
#There is a substantial amount of further work needed to either certify the quality of the source 
#of random numbers 
#(cf. NIST SP 800-90B, Recommendation for the Entropy Sources Used for Random Bit Generation) or 
#to use random variates within quantum algorithms (cf. uncertainty_models within Qiskit Aqua).